# 長い文書を分割して検索し、文書単位にまとめる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/21_search/long_document_retrieval.ipynb)

NLP クックブック 21-16 ／ 検索

埋め込みモデルには入力長の上限があり、長い文書は先頭しか見てもらえない。文書の後半に書かれた内容でも検索に当たるようにしたい。

In [1]:
import io, re, tarfile, urllib.request
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

def load_livedoor():
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

df = load_livedoor()
df = df[df["text"].str.len().between(1500, 4000)].sample(200, random_state=0).reset_index(drop=True)
split = lambda t: [s.strip() for s in re.split(r"(?<=[。！？])|\n+", t) if s and len(s.strip()) > 15]
sents = [split(t) for t in df["text"]]

# クエリ: 各文書の「後ろ 3 分の 1」にある 1 文 (正解はその文書)。文書の後半の内容を探す状況を模している
rng = np.random.default_rng(0)
queries = [s[int(rng.integers(len(s) * 2 // 3, len(s)))] for s in sents]
print(len(df), "文書 / 平均", int(df["text"].str.len().mean()), "文字 |", queries[0][:50])

200 文書 / 平均 2110 文字 | ■Eee Slate B121製品情報


In [2]:
model = SentenceTransformer("intfloat/multilingual-e5-small")
model.max_seq_length = 256
enc = lambda texts, prefix: model.encode([prefix + t for t in texts], normalize_embeddings=True, batch_size=32, show_progress_bar=False)
q_emb = enc(queries, "query: ")

# 方法 1: 文書をそのまま埋め込む (上限を超えた分は切り捨てられ、先頭しか反映されない)
whole = enc(df["text"].tolist(), "passage: ")
rank_whole = np.argsort(-(q_emb @ whole.T), axis=1)

# 方法 2: 約 300 文字のチャンクに分けて埋め込み、文書のスコア = その文書のチャンクの最大スコア
chunks, owner = [], []
for doc_id, ss in enumerate(sents):
    cur = ""
    for s in ss + [None]:
        if s is None or len(cur) + len(s) > 300:
            if cur:
                chunks.append(cur); owner.append(doc_id)
            cur = ""
        if s:
            cur += s
owner = np.array(owner)
chunk_emb = enc(chunks, "passage: ")
sims = q_emb @ chunk_emb.T
doc_scores = np.stack([sims[:, owner == d].max(axis=1) for d in range(len(df))], axis=1)
rank_chunk = np.argsort(-doc_scores, axis=1)

def report(name, ranking):
    rank = np.array([np.where(ranking[i] == i)[0][0] + 1 for i in range(len(ranking))])
    print(f"{name:28} Recall@1 = {(rank <= 1).mean():.3f}  Recall@5 = {(rank <= 5).mean():.3f}  MRR = {(1 / rank).mean():.3f}")

print(len(chunks), "チャンク")
report("文書全体を 1 つの埋め込みに", rank_whole)
report("チャンクに分けて最大値", rank_chunk)

1527 チャンク
文書全体を 1 つの埋め込みに              Recall@1 = 0.300  Recall@5 = 0.480  MRR = 0.393
チャンクに分けて最大値                  Recall@1 = 0.700  Recall@5 = 0.845  MRR = 0.769


- 入力長の上限を超えた部分は黙って切り捨てられる。エラーにならないので、気づかないまま「後半が検索に当たらない」状態になりやすい。
- チャンクに分けて、文書のスコアをチャンクの最大値（または上位数個の和）とするのが基本の対処。どのチャンクが当たったかも分かるので、該当箇所を示せる。
- この実験のクエリは文書中の文そのものなので、数値は実際より高く出る。差の向きを見るための簡易な確認である。
- 上限の長い埋め込みモデル（8192 トークンなど）もあるが、長い文書を 1 つのベクトルに押し込むと細部が薄まる。長さの上限が大きくても、分割は有効なことが多い。